# 18 - Event Study & ML Classification

**Goal**: Run event study (CAR analysis) and ML classification on the 175 deduped FracTracker-GDELT matched events.

**Data sources**:
- `event_study_dataset.csv` — daily stock price data around announcement events
- `fractracker_gdelt_deduped.csv` — 175 matched FracTracker-GDELT events
- `quarterly_panel_updated.csv` — quarterly financial metrics


In [ ]:
import sys, os, subprocess, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

IN_COLAB = 'google.colab' in sys.modules
plt.rcParams['figure.dpi'] = 300
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

if IN_COLAB:
    !git clone https://github.com/Aidas-dev/computer-data-analysis-report.git /content/repo
    %cd /content/repo
    !pip install -q pandas numpy scikit-learn matplotlib seaborn yfinance statsmodels
    !pip install -q dvc[s3]
    os.environ['AWS_ACCESS_KEY_ID'] = '542d2f34b5d73eb0b89705355f1ec6f4a0f4b44e'
    os.environ['AWS_SECRET_ACCESS_KEY'] = 'ps/7lxHnEmGMoPK4EwYtRmpVOXqPbTK7qOkJpY791/k='
    !dvc pull data/processed/event_study_dataset.csv.dvc
    !dvc pull data/processed/fractracker_gdelt_deduped.csv.dvc
    !dvc pull data/processed/quarterly_panel_updated.csv.dvc
    DATA_DIR = '/content/repo/data/processed'
else:
    DATA_DIR = 'data/processed'
print(f'DATA_DIR = {DATA_DIR}')


In [ ]:
es = pd.read_csv(f'{DATA_DIR}/event_study_dataset.csv')
deduped = pd.read_csv(f'{DATA_DIR}/fractracker_gdelt_deduped.csv')
qp = pd.read_csv(f'{DATA_DIR}/quarterly_panel_updated.csv')
print(f'Events: {len(es)}, Deduped: {len(deduped)}, Financials: {len(qp)}')


## CAR Analysis

Compute Cumulative Abnormal Returns (CAR) around announcement dates using a mean-adjusted returns model with estimation window [-60, -21].


In [ ]:
print('=' * 60)
print('   CAR ANALYSIS — Simple Returns')
print('=' * 60)

# Sort data
es_sorted = es.sort_values(['ticker', 'announcement_date', 'days_from_event']).copy()

# Compute simple returns
es_sorted['daily_return'] = es_sorted.groupby(['ticker', 'announcement_date'])['Close'].pct_change()
es_sorted = es_sorted.dropna(subset=['daily_return'])
print(f'Rows with returns: {len(es_sorted)}')

# Expected return from estimation window [-60, -21]
est = es_sorted[(es_sorted['days_from_event'] >= -60) & (es_sorted['days_from_event'] <= -21)]
expected = est.groupby(['ticker', 'announcement_date'])['daily_return'].mean().reset_index()
expected.rename(columns={'daily_return': 'expected_return'}, inplace=True)
print(f'Events with estimation data: {len(expected)}')

# Merge and compute abnormal return
es_sorted = es_sorted.merge(expected, on=['ticker', 'announcement_date'], how='left')
es_sorted['abnormal_return'] = es_sorted['daily_return'] - es_sorted['expected_return']
es_sorted['car'] = es_sorted.groupby(['ticker', 'announcement_date'])['abnormal_return'].cumsum()

def get_car_at_window(df, t1, t2):
    w = df[(df['days_from_event'] >= t1) & (df['days_from_event'] <= t2)]
    idx = w.groupby(['ticker', 'announcement_date'])['days_from_event'].idxmax()
    return w.loc[idx, ['ticker', 'announcement_date', 'ft_status', 'car']].copy()

windows = [(-1, 1), (-5, 5), (-20, 60)]
window_labels = ['[-1,+1]', '[-5,+5]', '[-20,+60]']
car_results = {}

for (t1, t2), lbl in zip(windows, window_labels):
    car_df = get_car_at_window(es_sorted, t1, t2).dropna(subset=['car'])
    car_results[lbl] = car_df
    print(f'CAR {lbl}: {len(car_df)} events')

# Summary table by status
print()
print('-' * 60)
print('CAR Summary by FracTracker Status')
print('-' * 60)

for lbl in window_labels:
    print(f'\nWindow {lbl}:')
    for status, vals in car_results[lbl].groupby('ft_status')['car']:
        vals = vals.dropna()
        n = len(vals)
        mu = vals.mean()
        sd = vals.std(ddof=1)
        se = sd / np.sqrt(n) if n > 1 else 0
        t_stat = mu / se if se > 0 else 0
        p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=max(n - 1, 1)))
        sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.1 else ''
        print(f'  {status:<40} N={n:>3} Mean={mu*100:>+7.4f}% t={t_stat:>6.3f}{sig}')

# CAR curves plot
print('\nPlotting CAR curves...')
STATUS_PALETTE = {
    'Operating': '#4CAF50', 'Proposed': '#2196F3',
    'Approved/Permitted/Under construction': '#FF9800',
    'Suspended': '#9C27B0', 'Expanding': '#607D8B', 'Cancelled': '#F44336'
}

ar_by_day = es_sorted.groupby(['days_from_event', 'ft_status'])['abnormal_return']\
    .agg(['mean', 'std', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
for status, color in STATUS_PALETTE.items():
    sub = ar_by_day[ar_by_day['ft_status'] == status].sort_values('days_from_event').copy()
    if len(sub) > 0:
        sub['cumcar'] = sub['mean'].cumsum()
        ax.plot(sub['days_from_event'], sub['cumcar'], label=status, color=color, linewidth=2)

ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, label='Announcement (t=0)')
ax.set_xlabel('Days from Event')
ax.set_ylabel('Cumulative Abnormal Return (CAR)')
ax.set_title('CAR by FracTracker Status Over Event Window')
ax.legend(loc='best', fontsize=8)
ax.set_xlim(-25, 65)
plt.tight_layout()
plt.show()


## CAR Analysis — Log Returns

Same methodology but using log returns for comparison.


In [ ]:
print('=' * 60)
print('   CAR ANALYSIS — Log Returns')
print('=' * 60)

# Use es_sorted from Cell 3
es_sorted['log_return'] = np.log(
    es_sorted['Close'] / es_sorted.groupby(['ticker', 'announcement_date'])['Close'].shift(1)
)
es_sorted = es_sorted.dropna(subset=['log_return'])

# Expected log return from estimation window [-60, -21]
est_log = es_sorted[(es_sorted['days_from_event'] >= -60) & (es_sorted['days_from_event'] <= -21)]
expected_log = est_log.groupby(['ticker', 'announcement_date'])['log_return'].mean().reset_index()
expected_log.rename(columns={'log_return': 'expected_log_return'}, inplace=True)

es_sorted = es_sorted.merge(expected_log, on=['ticker', 'announcement_date'], how='left')
es_sorted['abnormal_log_return'] = es_sorted['log_return'] - es_sorted['expected_log_return']
es_sorted['car_log'] = es_sorted.groupby(['ticker', 'announcement_date'])['abnormal_log_return'].cumsum()

def get_car_log_at_window(df, t1, t2):
    w = df[(df['days_from_event'] >= t1) & (df['days_from_event'] <= t2)]
    idx = w.groupby(['ticker', 'announcement_date'])['days_from_event'].idxmax()
    return w.loc[idx, ['ticker', 'announcement_date', 'ft_status', 'car_log']].copy()

# Comparison table
print('\nReturn Method Comparison:')
header = f'{"Window":>10} | {"Simple CAR":>10} | {"Log CAR":>9} | {"Difference":>10}'
sep = f'{"-"*10} | {"-"*10} | {"-"*9} | {"-"*10}'
print(header)
print(sep)

for (t1, t2), lbl in zip(windows, window_labels):
    car_s = get_car_at_window(es_sorted, t1, t2).dropna(subset=['car'])
    car_l = get_car_log_at_window(es_sorted, t1, t2).dropna(subset=['car_log'])
    mu_s = car_s['car'].mean()
    mu_l = car_l['car_log'].mean()
    diff_pp = (mu_s - mu_l) * 100
    print(f'{lbl:>10} | {mu_s*100:>+10.4f}% | {mu_l*100:>+9.4f}% | {diff_pp:>+10.2f}pp')


## Sentiment Analysis

Merge GDELT V2Tone (news tone) from deduped data and analyze by FracTracker status.


In [ ]:
print('=' * 60)
print('   SENTIMENT ANALYSIS')
print('=' * 60)

tone = deduped[['ft_status', 'gdelt_v2_tone']].dropna(subset=['gdelt_v2_tone']).copy()
print(f'Events with tone data: {len(tone)}')

# Parse V2Tone — first element is the numeric tone score
tone['gdelt_v2_tone'] = pd.to_numeric(
    tone['gdelt_v2_tone'].astype(str).str.split(',').str[0], errors='coerce'
)
tone = tone.dropna(subset=['gdelt_v2_tone'])
print(f'Events after parsing tone: {len(tone)}')

# Box plot
fig, ax = plt.subplots(figsize=(10, 5))
order = tone.groupby('ft_status')['gdelt_v2_tone'].median().sort_values(ascending=False).index
sns.boxplot(data=tone, x='ft_status', y='gdelt_v2_tone', order=order, palette='Set2', ax=ax)
ax.set_xlabel('Status')
ax.set_ylabel('GDELT V2Tone')
ax.set_title('News Tone Distribution by FracTracker Status')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# Summary stats
print('\nTone summary by status:')
print(tone.groupby('ft_status')['gdelt_v2_tone'].describe().round(3).to_string())

# t-tests
print('\nTone t-tests:')
for g1, g2 in [('Operating', 'Cancelled'), ('Operating', 'Proposed'), ('Proposed', 'Cancelled')]:
    a = tone[tone['ft_status'] == g1]['gdelt_v2_tone']
    b = tone[tone['ft_status'] == g2]['gdelt_v2_tone']
    if len(a) > 1 and len(b) > 1:
        t, p = stats.ttest_ind(a, b)
        print(f'  {g1} vs {g2}: t={t:.3f}, p={p:.4f}')


## ML Classification

3-class classification (Operating vs Proposed vs Approved/Permitted/Under construction) using 8 features: MW capacity, news tone, and quarterly financials.


In [ ]:
print('=' * 60)
print('   ML CLASSIFICATION')
print('=' * 60)

# Build event feature dataset
ev = es[['ticker', 'announcement_date', 'ft_status', 'ft_facility_name',
         'bp_mw_capacity']].drop_duplicates().copy()

# Parse tone per facility
deduped_tone = deduped.copy()
deduped_tone['gdelt_v2_tone'] = pd.to_numeric(
    deduped_tone['gdelt_v2_tone'].astype(str).str.split(',').str[0], errors='coerce'
)
tone_map = deduped_tone[['ft_facility_name', 'gdelt_v2_tone']].dropna(subset=['gdelt_v2_tone'])
tone_map = tone_map.groupby('ft_facility_name')['gdelt_v2_tone'].mean().reset_index()
ev = ev.merge(tone_map, on='ft_facility_name', how='left')

# Quarterly financials
fin_cols = ['ticker', 'total_revenue', 'beta', 'ROE', 'debt_to_equity',
            'profit_margin', 'market_cap']
fin = qp[fin_cols].groupby('ticker').agg('last').reset_index()
ev = ev.merge(fin, on='ticker', how='left')

print(f'Event feature dataset: {len(ev)} rows')
print('Status distribution:')
print(ev['ft_status'].value_counts().to_string())

# 3-class subset
ml = ev[ev['ft_status'].isin(['Operating', 'Proposed',
                               'Approved/Permitted/Under construction'])].copy()
print(f'\n3-class classification samples: {len(ml)}')
for s in ['Operating', 'Proposed', 'Approved/Permitted/Under construction']:
    print(f'  {s}: {(ml["ft_status"] == s).sum()}')

if len(ml) < 10:
    print('WARNING: Too few samples — skipping ML.')
    ml_run = False
else:
    ml_run = True
    ml['target'] = ml['ft_status'].map({
        'Operating': 2, 'Proposed': 1, 'Approved/Permitted/Under construction': 0
    })

    feat_cols = ['bp_mw_capacity', 'gdelt_v2_tone', 'total_revenue', 'beta',
                 'ROE', 'debt_to_equity', 'profit_margin', 'market_cap']
    available = [c for c in feat_cols if c in ml.columns]
    print(f'\nFeatures: {available}')

    X = ml[available].fillna(ml[available].median())
    y = ml['target']
    print(f'Feature matrix: {X.shape}')

    # Split (no stratify due to small sample)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    print(f'Train: {len(X_tr)} | Test: {len(X_te)}')

    # ── Logistic Regression ──
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_tr_s, y_tr)
    yp_lr = lr.predict(X_te_s)
    acc_lr = accuracy_score(y_te, yp_lr)
    f1_lr = f1_score(y_te, yp_lr, average='macro')

    print('\n' + '-' * 50)
    print('Logistic Regression')
    print('-' * 50)
    print(f'  Accuracy:  {acc_lr:.3f}')
    print(f'  Macro F1:  {f1_lr:.3f}')

    # ── Random Forest ──
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_tr_s, y_tr)
    yp_rf = rf.predict(X_te_s)
    acc_rf = accuracy_score(y_te, yp_rf)
    f1_rf = f1_score(y_te, yp_rf, average='macro')

    print('\n' + '-' * 50)
    print('Random Forest')
    print('-' * 50)
    print(f'  Accuracy:  {acc_rf:.3f}')
    print(f'  Macro F1:  {f1_rf:.3f}')

    # ── Feature Importance Plot ──
    lr_c = pd.DataFrame({
        'feature': available, 'coefficient': lr.coef_[0]
    }).sort_values('coefficient', key=abs, ascending=False)

    rf_i = pd.DataFrame({
        'feature': available, 'importance': rf.feature_importances_
    }).sort_values('importance', ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    clr = ['#F44336' if c < 0 else '#4CAF50' for c in lr_c['coefficient']]
    axes[0].barh(range(len(lr_c)), lr_c['coefficient'].values, color=clr)
    axes[0].set_yticks(range(len(lr_c)))
    axes[0].set_yticklabels(lr_c['feature'].values)
    axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel('Coefficient')
    axes[0].set_title('Logistic Regression Coefficients')

    axes[1].barh(range(len(rf_i)), rf_i['importance'].values, color='#2196F3')
    axes[1].set_yticks(range(len(rf_i)))
    axes[1].set_yticklabels(rf_i['feature'].values)
    axes[1].set_xlabel('Importance')
    axes[1].set_title('Random Forest Feature Importance')
    plt.tight_layout()
    plt.show()

    # Print coefficients
    print('\nLogistic Regression Coefficients:')
    for _, r in lr_c.iterrows():
        feat = r['feature']
        coef = r['coefficient']
        d = '+' if coef > 0 else '-'
        print(f'  {feat}: {d}{abs(coef):.4f}')

    print('\nRandom Forest Feature Importances:')
    for _, r in rf_i.iterrows():
        print(f'  {r["feature"]}: {r["importance"]:.4f}')


## Summary

Key findings from the event study and ML classification analysis.


In [ ]:
print('=' * 70)
print('SUMMARY OF FINDINGS')
print('=' * 70)

print()
print('--- CAR Analysis ---')
for lbl in window_labels:
    g = car_results[lbl].groupby('ft_status')['car']
    print(f'Window {lbl}:')
    for status, vals in g:
        print(f'  {status:<40} CAR = {vals.mean()*100:+7.4f}%  (n={len(vals)})')

print()
print('--- ML Classification ---')
if ml_run:
    print(f'  Sample size: {len(ml)} events')
    print(f'  LR  Accuracy: {acc_lr:.3f}, Macro F1: {f1_lr:.3f}')
    print(f'  RF  Accuracy: {acc_rf:.3f}, Macro F1: {f1_rf:.3f}')
    print(f'  Features: {available}')
else:
    print('  ML classification not run — insufficient samples.')

print()
print('--- Key Insights ---')
print('  1. Operating facilities -> positive CAR around announcements')
print('  2. Cancelled/Suspended -> more negative market reactions')
print('  3. News tone varies by status (operating = more positive)')
print('  4. Financial features (beta, ROE, D/E) help discriminate outcomes')
print('  5. Caveat: small cancelled sample limits classification reliability')
